# MLOps: Machine Learning in Production

## 🎯 What You'll Learn

This series covers the complete MLOps lifecycle:
1. **Experiment Tracking** - MLflow, Weights & Biases
2. **API Development** - FastAPI for model serving
3. **Deployment** - Docker, cloud platforms
4. **Monitoring** - Observability and alerts
5. **CI/CD** - Automated ML pipelines
6. **Production Best Practices** - Scalability, reliability

## Why MLOps?

**The Problem:** Most ML models never make it to production
- Jupyter notebooks are great for experimentation
- But production requires different skills
- Need to handle scale, reliability, monitoring

**The Solution:** MLOps practices
- Version control for models and data
- Automated testing and deployment
- Continuous monitoring and improvement

## The MLOps Lifecycle

```
┌─────────────────────────────────────────────────────┐
│                   ML LIFECYCLE                      │
├─────────────────────────────────────────────────────┤
│                                                     │
│  Data → Experiment → Train → Evaluate → Deploy     │
│   ↑                                          ↓      │
│   └────────────── Monitor & Retrain ─────────┘      │
│                                                     │
└─────────────────────────────────────────────────────┘
```

### Key Components

1. **Experiment Tracking**
   - Log hyperparameters, metrics, artifacts
   - Compare different runs
   - Reproduce experiments

2. **Model Registry**
   - Version control for models
   - Track model lineage
   - Manage staging/production models

3. **Model Serving**
   - REST APIs (FastAPI, Flask)
   - Batch predictions
   - Real-time inference

4. **Monitoring**
   - Model performance metrics
   - Data drift detection
   - System health (latency, errors)

## Quick Demo: Simple ML API

Let's build a minimal model serving API:

In [ ]:
# Install dependencies (uncomment to run)
# !pip install fastapi uvicorn scikit-learn joblib

**Model Training (.fit()):** The `.fit()` method is where the model learns from data. It adjusts the model's internal parameters to minimize prediction errors on the training data.

For different model types, `.fit()` does different things:
- **Linear models**: Finds the best-fit line/plane (minimizes squared error)
- **Decision trees**: Recursively splits data to separate classes/values
- **Neural networks**: Runs gradient descent over many epochs
- **Transformers (StandardScaler, PCA)**: Computes statistics (mean, variance, components) from the training data

**Random Forest:** An ensemble method that builds many decision trees and averages their predictions (regression) or takes a majority vote (classification).

Two key tricks make it powerful:
1. **Bagging**: Each tree trains on a random subset of the data (with replacement)
2. **Feature randomness**: Each split considers only a random subset of features

This decorrelates the trees, reducing overfitting. Random Forests handle non-linear relationships, missing values, and mixed feature types well, with minimal tuning needed.

**Random Seed:** Setting a seed ensures **reproducibility** — the same random numbers are generated each time the code runs. This is critical in ML experiments because:
- Train/test splits will be the same
- Weight initialization will be identical
- Any randomized algorithm (dropout, data augmentation) will behave consistently

Without a fixed seed, your results would vary between runs, making it impossible to debug or compare experiments.

In [ ]:
# Simple model training
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
import joblib

# Train model
iris = load_iris()
model = RandomForestClassifier(n_estimators=10, random_state=42)
model.fit(iris.data, iris.target)

# Save model
joblib.dump(model, 'iris_model.pkl')
print("✓ Model trained and saved")

**Model Prediction (.predict()):** After training, `.predict()` applies the learned model to new (unseen) data to generate predictions.

- **Classification**: Returns predicted class labels
- **Regression**: Returns predicted continuous values

The quality of predictions depends on how well the model was trained and whether the new data resembles the training distribution.

In [ ]:
# Create FastAPI application
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

app = FastAPI(title="Iris Classifier API")

# Load model
model = joblib.load('iris_model.pkl')

class PredictionRequest(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

class PredictionResponse(BaseModel):
    species: str
    confidence: float

@app.post("/predict", response_model=PredictionResponse)
async def predict(request: PredictionRequest):
    # Prepare features
    features = np.array([[
        request.sepal_length,
        request.sepal_width,
        request.petal_length,
        request.petal_width
    ]])
    
    # Predict
    prediction = model.predict(features)[0]
    probabilities = model.predict_proba(features)[0]
    
    species_names = ['setosa', 'versicolor', 'virginica']
    
    return PredictionResponse(
        species=species_names[prediction],
        confidence=float(probabilities[prediction])
    )

@app.get("/health")
async def health():
    return {"status": "healthy"}

print("✓ API created")
print("Run with: uvicorn main:app --reload")

## Testing the API

To test this API:

```bash
# Start server
uvicorn main:app --reload

# Test with curl
curl -X POST "http://localhost:8000/predict" \
  -H "Content-Type: application/json" \
  -d '{"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}'
```

Or visit: http://localhost:8000/docs for interactive API documentation

## What's Next?

In the following notebooks, you'll learn:

1. **Experiment Tracking** - Track and compare models
2. **FastAPI Deep Dive** - Advanced API features
3. **Model Deployment** - Production serving strategies
4. **Docker** - Containerize your applications
5. **Monitoring** - Observe model behavior in production
6. **CI/CD** - Automate your ML pipeline
7. **Cloud Deployment** - Deploy to AWS/Azure/GCP

🚀 Ready to take ML to production!